In [1]:
from __future__ import annotations

import json
import logging
import re
import subprocess
import types
from pathlib import Path

import pandas as pd
import soccerdata as sd
import soccerdata._common as common
import undetected_chromedriver as uc
import yaml
from pymongo import MongoClient

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "dev_scripts" else Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
CHROME_PATH = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("game_missing_players")

print(PROJECT_ROOT)

[06/28/26 23:19:51] INFO     No custom team name replacements found. You can configure these in       ]8;id=8172147;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=8172148;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/mario_omescu/soccerdata/config/teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=8172154;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=8172155;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/mario_omescu/soccerdata/config/league_dict.json.                               

/Users/mario_omescu/Library/Mobile Documents/com~apple~CloudDocs/Sports Analytics/WhoScored Events Data


In [2]:
def get_chrome_major(path: str = CHROME_PATH) -> int:
    out = subprocess.check_output([path, "--version"], text=True)
    match = re.search(r"(\d+)\.", out)
    if not match:
        raise RuntimeError(f"Could not detect Chrome version from: {out}")
    return int(match.group(1))


def patch_soccerdata_chromedriver() -> None:
    def patched_init_webdriver(self):
        chrome_path = str(self.path_to_browser or CHROME_PATH)
        chrome_major = get_chrome_major(chrome_path)

        opts = uc.ChromeOptions()
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-dev-shm-usage")
        opts.add_argument("--start-maximized")

        return uc.Chrome(
            options=opts,
            version_main=chrome_major,
            browser_executable_path=chrome_path,
            headless=self.headless,
        )

    common.BaseSeleniumReader._init_webdriver = patched_init_webdriver


def patch_soccerdata_json_loader() -> None:
    def tolerant_json_load(fp, *args, **kwargs):
        content = fp.read()
        if isinstance(content, bytes):
            content = content.decode("utf-8", errors="ignore")

        content = content.strip()
        if content.startswith("<html"):
            start = content.find("{")
            end = content.rfind("}") + 1
            if start != -1 and end > start:
                content = content[start:end]

        return json.loads(content)

    json.load = tolerant_json_load


patch_soccerdata_chromedriver()
patch_soccerdata_json_loader()

print(f"SoccerData notebook patches applied. Chrome major: {get_chrome_major()}")

SoccerData notebook patches applied. Chrome major: 149


In [12]:
MATCH_ID = 1910894

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

season_cfg = cfg["season"]
mongo_cfg = cfg["mongo"]
collections = mongo_cfg["collections"]

SEASON_YEAR = season_cfg["year"]
SEASON_SHORT = season_cfg["year_short"]
LEAGUE = season_cfg["league"]

client = MongoClient(mongo_cfg["url"])
db = client[mongo_cfg["db"]]
collection_schedule = db[collections["collection_schedule"]]
collection_teams = db[collections["collection_teams"]]

available_teams = pd.DataFrame(collection_teams.find({},{'_id': 0, 'ws_team_id': 1, 'ws_team_name': 1}))
available_teams.rename(columns=({'ws_team_id': 'team_id', 'ws_team_name': 'team_name'}), inplace=True)

schedule_doc = collection_schedule.find_one({"game_id": MATCH_ID}, {"_id": 0})
client.close()

if schedule_doc is None:
    raise ValueError(f"No game_schedule document found for game_id={MATCH_ID}")

display(pd.DataFrame([schedule_doc]))

,game_id,attendance,away_goals,away_manager,away_team_id,away_team_name,competition_country,competition_name,game_date,game_status,home_goals,home_manager,home_team_id,home_team_name,referee,season,venue,week
0,1910894,30210,1,Merlin Polzin,38,Hamburger SV,Germany,Bundesliga,2026-05-16 14:30:00,finished,1,Kasper Hjulmand,36,Bayer Leverkusen,Tobias Stieler,2025-2026,BayArena,34


In [4]:
ws = sd.WhoScored(
    leagues=LEAGUE,
    seasons=SEASON_YEAR,
    headless=True,
    path_to_browser=CHROME_PATH,
    no_cache=False,
)

def one_match_schedule(self, force_cache=False):
    return pd.DataFrame([{
        "league": LEAGUE,
        "season": SEASON_SHORT,
        "game": f"match_{MATCH_ID}",
        "game_id": MATCH_ID,
        "home_team": schedule_doc["home_team_name"],
        "away_team": schedule_doc["away_team_name"],
    }])

ws.read_schedule = types.MethodType(one_match_schedule, ws)

print("WhoScored reader ready")

[06/28/26 23:20:07] INFO     Saving cached data to /Users/mario_omescu/soccerdata/data/WhoScored     ]8;id=8172162;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8172163;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

[06/28/26 23:20:08] INFO     patching driver executable /Users/mario_omescu/Library/Application      ]8;id=8172170;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py\patcher.py]8;;\:]8;id=8172171;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py#346\346]8;;\
                             Support/undetected_chromedriver/undetected_chromedriver                               

[06/28/26 23:20:10] INFO     setting properties for headless                                        ]8;id=8172178;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/__init__.py\__init__.py]8;;\:]8;id=8172179;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/__init__.py#493\493]8;;\

WhoScored reader ready


In [15]:
missing_players = ws.read_missing_players(match_id=MATCH_ID, force_cache=True)

if missing_players.empty:
    df_missing_players = pd.DataFrame(columns=[
        "game_id", "game_date", "week", "team", "player", "player_id", "reason", "status",
    ])
else:
    df_missing_players = missing_players.reset_index()
    df_missing_players["game_date"] = schedule_doc.get("game_date")
    df_missing_players["week"] = schedule_doc.get("week")
    df_missing_players["season"] = SEASON_YEAR
    df_missing_players.rename(columns={'team': 'team_name'}, inplace=True)
    df_missing_players = pd.merge(
        left=df_missing_players,
        right=available_teams,
        on='team_name',
        how='left'
    )

    
    ordered = [
        "game_id", "game_date", "season", "week", "team_id", "team",
        "player_id", "player",  "status", "reason",
    ]
    df_missing_players = df_missing_players[[c for c in ordered if c in df_missing_players.columns]]

print(f"Missing players rows: {len(df_missing_players)}")
display(df_missing_players)

[06/28/26 23:28:30] INFO     [1/1] Retrieving game with id=1910894                                 ]8;id=8172204;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=8172205;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#523\523]8;;\

Missing players rows: 8


,game_id,game_date,season,week,team_id,player_id,player,status,reason
0,1910894,2026-05-16 14:30:00,2025-2026,34,36,550465,Christian Kofane,Doubtful,injured doubtful
1,1910894,2026-05-16 14:30:00,2025-2026,34,36,329356,Martin Terrier,Out,injured
2,1910894,2026-05-16 14:30:00,2025-2026,34,36,395472,Nathan Tella,Doubtful,injured doubtful
3,1910894,2026-05-16 14:30:00,2025-2026,34,38,527354,Alexander Røssing-Lelesiit,Out,injured
4,1910894,2026-05-16 14:30:00,2025-2026,34,38,536129,Fernando Dickes,Out,injured
5,1910894,2026-05-16 14:30:00,2025-2026,34,38,234353,Jean-Luc Dompé,Doubtful,unfit
6,1910894,2026-05-16 14:30:00,2025-2026,34,38,369480,Nicolás Capaldo,Doubtful,injured doubtful
7,1910894,2026-05-16 14:30:00,2025-2026,34,38,136707,Robert Glatzel,Doubtful,injured doubtful


In [ ]:
try:
    ws.close()
except Exception:
    pass